In [1]:
import pygame
import random
import sys
import copy

# 1. 常量类 
class ChangLiang:
    宽度, 高度 = 450, 800
    方块大小 = 60
    槽位上限 = 7  
    帧率 = 60
    
    区域宽, 区域高 = 380, 400
    区域X, 区域Y = (宽度 - 380) // 2, 100
    
    颜色列表 = [
        (255, 80, 80), (160, 80, 255), (255, 150, 50), 
        (255, 230, 100), (80, 200, 255), (255, 100, 150), (100, 220, 100)
    ]
    背景顶色 = (200, 235, 200)
    背景底色 = (120, 180, 120)

# 2. 绘图函数 
def 画图标(画布, 颜色编号, 矩形, 是否被遮挡):
    中心 = 矩形.center
    半宽 = ChangLiang.方块大小 // 4
    颜色 = list(ChangLiang.颜色列表[颜色编号 % len(ChangLiang.颜色列表)])
    if 是否被遮挡: 颜色 = [值 // 2 for 值 in 颜色] # 被压住就变暗

    if 颜色编号 == 0: # 苹果
        
        pygame.draw.circle(画布, 颜色, (中心[0], 中心[1] + 2), 半宽)
        pygame.draw.rect(画布, (50, 150, 50), (中心[0] - 2, 中心[1] - 半宽 - 2, 4, 8))
        
    elif 颜色编号 == 1: # 葡萄
        
        for 偏移 in [(-6,-6), (6,-6), (0,0), (-6,6), (6,6)]:
            pygame.draw.circle(画布, 颜色, (中心[0] + 偏移[0], 中心[1] + 偏移[1]), 7)
            
    elif 颜色编号 == 2: # 胡萝卜
        
        顶点列表 = [(中心[0] - 10, 中心[1] - 12), (中心[0] + 10, 中心[1] - 12), (中心[0], 中心[1] + 18)]
        pygame.draw.polygon(画布, 颜色, 顶点列表)
        pygame.draw.rect(画布, (50, 200, 50), (中心[0] - 8, 中心[1] - 18, 16, 6), 0, 2)
        
    elif 颜色编号 == 3: # 麦穗
        
        for i in range(3):
            pygame.draw.ellipse(画布, 颜色, (中心[0] - 12, 中心[1] - 15 + i * 10, 24, 8))
            
    elif 颜色编号 == 4: # 水滴
        
        pygame.draw.circle(画布, 颜色, (中心[0], 中心[1] + 8), 10)
        pygame.draw.polygon(画布, 颜色, [(中心[0] - 10, 中心[1] + 8), (中心[0] + 10, 中心[1] + 8), (中心[0], 中心[1] - 15)])
        
    elif 颜色编号 == 5: # 橙子
        
        pygame.draw.circle(画布, 颜色, 中心, 半宽 + 2)
        pygame.draw.circle(画布, (255, 255, 255), 中心, 半宽 - 2, 1)
        
    elif 颜色编号 == 6: # 樱桃
        
        pygame.draw.circle(画布, 颜色, (中心[0] - 8, 中心[1] + 5), 7)
        pygame.draw.circle(画布, 颜色, (中心[0] + 8, 中心[1] + 5), 7)
        pygame.draw.lines(画布, (50, 150, 50), False, [(中心[0], 中心[1] - 10), (中心[0] - 8, 中心[1] + 5)], 2)
        
    else: # 兜底方案：画个方块
        pygame.draw.rect(画布, 颜色, (中心[0] - 半宽, 中心[1] - 半宽, 半宽 * 2, 半宽 * 2), 0, 4)

# 3. 方块类
class FangKuai:
    def __init__(self, 编号, 颜色索引, 层级, x, y):
        self.id = 编号
        self.color_id = 颜色索引
        self.layer = 层级
        self.更新位置(x, y)

    def 更新位置(self, x, y):
        self.x, self.y = x, y
        # 根据层级制造立体偏移
        偏移X = self.layer * 4
        偏移Y = -self.layer * 4
        self.rect = pygame.Rect(x + 偏移X, y + 偏移Y, ChangLiang.方块大小, ChangLiang.方块大小)
        self.is_covered = False

    def 绘制方块(self, 屏幕): 
        # 画阴影
        pygame.draw.rect(屏幕, (40, 70, 40), (self.rect.x+3, self.rect.y+3, self.rect.width, self.rect.height), 0, 10)
        # 画底座
        底座颜色 = (255, 255, 250) if not self.is_covered else (160, 170, 160)
        pygame.draw.rect(屏幕, 底座颜色, self.rect, 0, 10)
        # 画边框
        边框颜色 = ChangLiang.颜色列表[self.color_id % 7] if not self.is_covered else (80, 90, 80)
        pygame.draw.rect(屏幕, 边框颜色, self.rect, 2, 10)
        # 画具体图标
        画图标(屏幕, self.color_id, self.rect, self.is_covered)

# 4. 管理类
class GuanLiQi:
    def __init__(self):
        self.所有方块 = []
        self.槽位 = []
        self.历史记录 = []
        self.初始化游戏()

    def 初始化游戏(self):
        self.所有方块 = []
        self.槽位 = []
        self.历史记录 = []
        卡池 = []
        for i in range(20): 卡池.extend([i % 7] * 3) # 确保成对
        random.shuffle(卡池)

        for i, 颜色索引 in enumerate(卡池):
            层 = random.randint(0, 4)
            x = random.randint(ChangLiang.区域X, ChangLiang.区域X + ChangLiang.区域宽 - ChangLiang.方块大小)
            y = random.randint(ChangLiang.区域Y, ChangLiang.区域Y + ChangLiang.区域高 - ChangLiang.方块大小)
            self.所有方块.append(FangKuai(i, 颜色索引, 层, x, y))
        self.更新遮挡逻辑()

    def 更新遮挡逻辑(self):
        self.所有方块.sort(key=lambda c: c.layer)
        for i, 方块1 in enumerate(self.所有方块):
            方块1.is_covered = False
            for j in range(i + 1, len(self.所有方块)):
                if 方块1.rect.colliderect(self.所有方块[j].rect):
                    方块1.is_covered = True
                    break

    def 保存进度(self):
        self.历史记录.append((copy.deepcopy(self.所有方块), copy.deepcopy(self.槽位)))
        if len(self.历史记录) > 10: self.历史记录.pop(0)

    def 处理点击(self, 点击位置):
        for i in range(len(self.所有方块) - 1, -1, -1):
            方块 = self.所有方块[i]
            if 方块.rect.collidepoint(点击位置) and not 方块.is_covered:
                self.保存进度()
                被点中的方块 = self.所有方块.pop(i)
                self.槽位.append(被点中的方块)
                self.消除检测()
                self.更新遮挡逻辑()
                return True
        return False

    def 消除检测(self):
        if len(self.槽位) < 3: return
        i = 0
        while i <= len(self.槽位) - 3:
            if self.槽位[i].color_id == self.槽位[i+1].color_id == self.槽位[i+2].color_id:
                del self.槽位[i:i+3]
                i = 0 
            else:
                i += 1

    def 撤回操作(self):
        if self.历史记录:
            self.所有方块, self.槽位 = self.历史记录.pop()
            self.更新遮挡逻辑()

    def 道具_洗牌(self):
        if not self.所有方块: return
        self.保存进度()
        for 方块 in self.所有方块:
            新X = random.randint(ChangLiang.区域X, ChangLiang.区域X + ChangLiang.区域宽 - ChangLiang.方块大小)
            新Y = random.randint(ChangLiang.区域Y, ChangLiang.区域Y + ChangLiang.区域高 - ChangLiang.方块大小)
            方块.更新位置(新X, 新Y)
        self.更新遮挡逻辑()

    def 道具_移出(self):
        if not self.槽位: return
        self.保存进度()
        移出的方块 = self.槽位[:3]
        self.槽位 = self.槽位[3:]
        for 方块 in 移出的方块:
            方块.layer = 5 
            新X = random.randint(ChangLiang.区域X, ChangLiang.区域X + ChangLiang.区域宽 - ChangLiang.方块大小)
            新Y = random.randint(ChangLiang.区域Y, ChangLiang.区域Y + ChangLiang.区域高 - ChangLiang.方块大小)
            方块.更新位置(新X, 新Y)
            self.所有方块.append(方块)
        self.更新遮挡逻辑()

    def 绘制全部(self, 屏幕):
        for 方块 in self.所有方块: 方块.绘制方块(屏幕)
        # 槽位UI
        槽位矩形 = pygame.Rect((ChangLiang.宽度-400)//2, 630, 400, 75)
        pygame.draw.rect(屏幕, (60, 90, 60), 槽位矩形, 0, 15)
        for i in range(ChangLiang.槽位上限):
            格位 = pygame.Rect(槽位矩形.x+10 + i*55, 槽位矩形.y+10, 50, 50)
            pygame.draw.rect(屏幕, (40, 60, 40), 格位, 0, 8)
            if i < len(self.槽位):
                临时方块 = copy.copy(self.槽位[i])
                临时方块.rect = 格位
                临时方块.is_covered = False
                临时方块.绘制方块(屏幕)

# 5. 入口函数
def 绘制渐变背景(屏幕):
    for y in range(ChangLiang.高度):
        比例 = y / ChangLiang.高度
        颜色 = [int(ChangLiang.背景顶色[i]*(1-比例) + ChangLiang.背景底色[i]*比例) for i in range(3)]
        pygame.draw.line(屏幕, 颜色, (0, y), (ChangLiang.宽度, y))

def 启动游戏():
    pygame.init()
    pygame.mixer.init()
    
    屏幕 = pygame.display.set_mode((ChangLiang.宽度, ChangLiang.高度))
    pygame.display.set_caption("养了个羊")
    时钟 = pygame.time.Clock()
    管理器 = GuanLiQi()
    
    # 字体与音乐
    字体 = pygame.font.Font("Songti.ttf", 18)
    
    #pygame.mixer.music.load("yang_bg.mp3") 
    #pygame.mixer.music.set_volume(0.03)
    #pygame.mixer.music.play(-1)
    

    按钮区域 = {
        "移出": pygame.Rect(50, 730, 80, 40),
        "撤回": pygame.Rect(185, 730, 80, 40),
        "洗牌": pygame.Rect(320, 730, 80, 40)
    }

    while True:
        绘制渐变背景(屏幕)
        
        for 事件 in pygame.event.get():
            if 事件.type == pygame.QUIT: pygame.quit(); sys.exit()
            if 事件.type == pygame.MOUSEBUTTONDOWN:
                if 按钮区域["移出"].collidepoint(事件.pos): 管理器.道具_移出()
                elif 按钮区域["撤回"].collidepoint(事件.pos): 管理器.撤回操作()
                elif 按钮区域["洗牌"].collidepoint(事件.pos): 管理器.道具_洗牌()
                else: 管理器.处理点击(事件.pos)

        管理器.绘制全部(屏幕)
        
        for 文本, 矩形 in 按钮区域.items():
            pygame.draw.rect(屏幕, (100, 160, 100), 矩形, 0, 10)
            pygame.draw.rect(屏幕, (255, 255, 255), 矩形, 2, 10)
            文字表面 = 字体.render(文本, True, (255, 255, 255))
            屏幕.blit(文字表面, (矩形.centerx - 文字表面.get_width()//2, 矩形.centery - 文字表面.get_height()//2))

        # 胜负判定
        if len(管理器.槽位) >= ChangLiang.槽位上限: 管理器.初始化游戏()
        elif not 管理器.所有方块 and not 管理器.槽位: 管理器.初始化游戏()

        pygame.display.flip()
        时钟.tick(ChangLiang.帧率)

if __name__ == "__main__":
    启动游戏()

pygame 2.6.1 (SDL 2.28.4, Python 3.12.2)
Hello from the pygame community. https://www.pygame.org/contribute.html


KeyboardInterrupt: 